In [1]:
import json

In [2]:
def read_json(path: str):
    with open(path, 'r') as f:
        return json.load(f)

In [3]:
path = '/home/abraham/uni/ikt453/project/v1/data/samples/schedule.json'
data = read_json(path)

In [5]:
30e3/len(data['games'])

24.193548387096776

In [8]:
browser_headers = """
accept
*/*
accept-encoding
gzip, deflate, br, zstd
accept-language
nb-NO,nb;q=0.9,no;q=0.8,nn;q=0.7,en-US;q=0.6,en;q=0.5
cache-control
no-cache
origin
https://www.nba.com
pragma
no-cache
priority
u=1, i
referer
https://www.nba.com/
sec-ch-ua
"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"
sec-ch-ua-mobile
?0
sec-ch-ua-platform
"Windows"
sec-fetch-dest
empty
sec-fetch-mode
cors
sec-fetch-site
same-site
user-agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36
"""

headers = {}
for idx, line in enumerate(browser_headers.strip().split('\n')):
    if idx % 2 == 0:
        key = line.strip()
    else:
        value = line.strip()
        headers[key] = value

def prettyprint(x):
    print(json.dumps(x, indent=4))

prettyprint(headers)

{
    "accept": "*/*",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "nb-NO,nb;q=0.9,no;q=0.8,nn;q=0.7,en-US;q=0.6,en;q=0.5",
    "cache-control": "no-cache",
    "origin": "https://www.nba.com",
    "pragma": "no-cache",
    "priority": "u=1, i",
    "referer": "https://www.nba.com/",
    "sec-ch-ua": "\"Chromium\";v=\"146\", \"Not-A.Brand\";v=\"24\", \"Google Chrome\";v=\"146\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-site",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}


In [9]:
import requests
url = 'https://stats.nba.com/stats/drafthistory?College=&LeagueID=00&OverallPick=&RoundNum=&RoundPick=&Season=&TeamID=0&TopX='
rsp = requests.get(url, headers=headers)
print(rsp.status_code)

200


In [ ]:
def write_json(path: str, data):
    with open(path, 'w') as f:
        json.dump(data, f, indent=4)

j = rsp.json()
# write_json('/home/abraham/uni/ikt453/project/v1/data/samples_nbaapi/drafthistory.json', j)

In [12]:
j = rsp.json()

In [16]:
list(j)

['resource', 'parameters', 'resultSets']

In [25]:
import pandas as pd
x = j['resultSets'][0]
x = pd.DataFrame(x['rowSet'], columns=x['headers'])
x.SEASON.agg(['min', 'max'])

min    1947
max    2025
Name: SEASON, dtype: object

In [27]:
list(j['resultSets'][0])

['name', 'headers', 'rowSet']

In [31]:
m = x.ROUND_PICK == 0
x[m]

,PERSON_ID,PLAYER_NAME,SEASON,ROUND_NUMBER,ROUND_PICK,OVERALL_PICK,DRAFT_TYPE,TEAM_ID,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,ORGANIZATION,ORGANIZATION_TYPE,PLAYER_PROFILE_FLAG
6630,76233,Bill Bradley,1965,0,0,0,Territorial,1610612752,New York,Knicks,NYK,Princeton,College/University,1
6631,76302,Bill Buntin,1965,0,0,0,Territorial,1610612765,Detroit,Pistons,DET,Michigan,College/University,1
6632,76832,Gail Goodrich,1965,0,0,0,Territorial,1610612747,Los Angeles,Lakers,LAL,California-Los Angeles,College/University,1
6732,76983,Walt Hazzard,1964,0,0,0,Territorial,1610612747,Los Angeles,Lakers,LAL,California-Los Angeles,College/University,1
6733,78579,George Wilson,1964,0,0,0,Territorial,1610612758,Cincinnati,Royals,CIN,Cincinnati,College/University,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8369,79320,Bob Jake,1947,0,0,0,Draft,1610610024,Baltimore,Bullets,BAL,Vermont,College/University,0
8370,79322,Charles Raynor,1947,0,0,0,Draft,1610610024,Baltimore,Bullets,BAL,Houston,College/University,0
8371,79323,John Rusinko,1947,0,0,0,Draft,1610610024,Baltimore,Bullets,BAL,Penn State,College/University,0
8372,76773,Harry Gallatin,1947,0,0,0,Draft,1610610024,Baltimore,Bullets,BAL,Truman State,College/University,1


In [36]:
pbp = 'https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_0022000180.json'
rsp = requests.get(pbp, headers=headers)
print(rsp.status_code)

200


In [37]:
rsp_json = rsp.json()
list(rsp_json)

['meta', 'game']

In [38]:
write_json('/home/abraham/uni/ikt453/project/v1/data/samples_nbaapi/playbyplay_0022000180.json', rsp_json)

In [41]:
a = pd.DataFrame(rsp_json['game']['actions'])

In [51]:
k = a[['actionType', 'subType']].value_counts()

indices, indexor = (
    k.index
    .sortlevel(0)
)

k = k.iloc[indexor]
k.index = indices
k

actionType     subType        
2pt            DUNK                7
               Hook                7
               Jump Shot          48
               Layup              50
3pt            Jump Shot          70
foul           offensive           2
               personal           32
               technical           1
freethrow      1 of 1              9
               1 of 2             13
               1 of 3              2
               2 of 2             13
               2 of 3              2
               3 of 3              2
game           end                 1
instantreplay  challenge           1
               request             1
jumpball       recovered           5
period         end                 4
               start               4
rebound        defensive          76
               offensive          33
stoppage       equipment issue     1
               out-of-bounds      11
substitution   in                 46
               out                46
timeout

In [59]:
params = a.groupby(['actionType', 'subType']).apply(lambda x: x.isna().sum(axis=0), include_groups=False)

In [ ]:
max_ = params.max(axis=1)
schema = (params != max_.values[:, None])


,subType,actionNumber,clock,timeActual,period,periodType,qualifiers,personId,x,y,...,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,turnoverTotal,stealPlayerName,stealPersonId,value,blockPlayerName,blockPersonId
actionType,,,,,,,,,,,,,,,,,,,,,
2pt,DUNK,True,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
2pt,Hook,True,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
2pt,Jump Shot,True,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,True,True
2pt,Layup,True,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,True,True
3pt,Jump Shot,True,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
foul,offensive,True,True,True,True,True,True,True,False,False,...,True,True,True,True,False,False,False,False,False,False
foul,personal,True,True,True,True,True,True,True,False,False,...,True,True,True,True,False,False,False,False,False,False
foul,technical,True,True,True,True,True,True,True,False,False,...,True,True,False,False,False,False,False,False,False,False
freethrow,1 of 1,True,True,True,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [70]:
leaders = 'https://stats.nba.com/stats/leagueleaders?ActiveFlag=&LeagueID=00&PerMode=Totals&Scope=S&Season=2019-20&SeasonType=Regular+Season&StatCategory=PTS'
rsp = requests.get(leaders, headers=headers)
print(rsp.status_code)

200


In [ ]:
# write_json('/home/abraham/uni/ikt453/project/v1/data/samples_nbaapi/leagueleaders_pts_2019-20.json', rsp.json())